In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)


g:\My Drive\Graduação e Pós\USP\MBA USP IA e Big Data\TCC\port_leadtime_prediction


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd

from src.io_utils import load_csv_files_from_dir
from src.processing.cleaning import (
    standardize_column_names,
    trim_string_columns,
    parse_datetime_columns,
)
from src.data_sources.port_call import load_port_call_files, process_port_call

In [3]:
RAW_ESTADIA_DIR = PROJECT_ROOT / "data" / "raw" / "estadia"
RAW_ESTADIA_DIR

WindowsPath('g:/My Drive/Graduação e Pós/USP/MBA USP IA e Big Data/TCC/port_leadtime_prediction/data/raw/estadia')

In [4]:
df_raw = load_port_call_files(RAW_ESTADIA_DIR)

print(df_raw.shape)
df_raw.head()

(162036, 23)


,DUV,Porto - Bitrigrama,Porto - Nome,Embarcacao - IMO,Embarcacao - Inscricao,Embarcacao - Nome,Tripulantes - Embarque,Tripulantes - Transito,Tripulantes - Desembarque,Tripulantes - Total,...,Passageiros - Total,Estadia - Motivos Atracacao,Estadia - Chegada no Porto,Estadia - Atracacao,Estadia - Desatracacao,Estadia - Saida do Porto,Porto Origem - Bitrigrama,Porto Origem - Nome,Porto Destino - Bitrigrama,Porto Destino - Nome
0,12022,BRSSZ,SANTOS ...,9634854,,ORION GLOBE,0.0,21,0,21,...,0,Carga ...,2022-01-13 06:00:00-03,2022-01-20 17:20:00-03,2022-01-23 21:52:00-03,2022-01-23 21:52:00-03,SGSIN,SINGAPURA (SINGAPORE) ...,SGSIN,SINGAPURA (SINGAPORE)
1,22022,BRSSZ,SANTOS ...,9699359,,COPENHAGEN EAGLE,0.0,19,0,19,...,0,Descarga ...,2022-01-15 09:05:00-03,2022-02-25 05:05:00-03,2022-02-28 13:45:00-03,2022-02-28 13:45:00-03,SGSIN,SINGAPURA (SINGAPORE) ...,BRPNG,PARANAGUA
2,32022,BRSSZ,SANTOS ...,9686273,,CEPHEUS OCEAN,0.0,19,0,19,...,0,Descarga ...,2022-01-19 10:00:00-03,2022-02-28 20:25:00-03,,,RU006,UST - LUGA ...,,
3,62022,BRRJ022,PORTO DO AÇU ...,9644158,,ZARAPITO,0.0,15,0,15,...,0,Off-shore ...,2022-01-01 08:00:00-03,2022-01-01 08:00:00-03,2022-01-03 10:27:00-03,2022-01-03 10:27:00-03,BRRIO,RIO DE JANEIRO ...,BRRIO,RIO DE JANEIRO
4,72022,BRMEA001,TERMINAL PETROBRÁS - MACAÉ - RJ ...,9618757,3813885801,MAR LIMPO II,0.0,13,0,13,...,0,"Desembarque/Embarque de Passageiros,Off-shore...",2022-01-01 17:45:00-03,2022-01-03 06:50:00-03,2022-01-04 03:00:00-03,2022-01-04 17:40:00-03,BRRIO,RIO DE JANEIRO ...,BRMEA001,TERMINAL PETROBRÁS - MACAÉ - RJ


In [5]:
df_port_call = process_port_call(df_raw)

print(df_port_call.shape)
df_port_call.head()

(162036, 23)


,port_call_id,port,port_name,imo,vessel_id,vessel_name,boarding_crew,transit_crew,unboarding_crew,total_crew,...,total_passengers,port_call_reasons,arrival_port_ts,berthing_ts,unberthing_ts,departure_port_ts,source_port,source_port_name,destination_port,destination_port_name
0,12022,BRSSZ,SANTOS,9634854,<NA>,ORION GLOBE,0.0,21,0,21,...,0,Carga,2022-01-13 06:00:00-03:00,2022-01-20 17:20:00-03:00,2022-01-23 21:52:00-03:00,2022-01-23 21:52:00-03:00,SGSIN,SINGAPURA (SINGAPORE),SGSIN,SINGAPURA (SINGAPORE)
1,22022,BRSSZ,SANTOS,9699359,<NA>,COPENHAGEN EAGLE,0.0,19,0,19,...,0,Descarga,2022-01-15 09:05:00-03:00,2022-02-25 05:05:00-03:00,2022-02-28 13:45:00-03:00,2022-02-28 13:45:00-03:00,SGSIN,SINGAPURA (SINGAPORE),BRPNG,PARANAGUA
2,32022,BRSSZ,SANTOS,9686273,<NA>,CEPHEUS OCEAN,0.0,19,0,19,...,0,Descarga,2022-01-19 10:00:00-03:00,2022-02-28 20:25:00-03:00,NaT,NaT,RU006,UST - LUGA,<NA>,<NA>
3,62022,BRRJ022,PORTO DO AÇU,9644158,<NA>,ZARAPITO,0.0,15,0,15,...,0,Off-shore,2022-01-01 08:00:00-03:00,2022-01-01 08:00:00-03:00,2022-01-03 10:27:00-03:00,2022-01-03 10:27:00-03:00,BRRIO,RIO DE JANEIRO,BRRIO,RIO DE JANEIRO
4,72022,BRMEA001,TERMINAL PETROBRÁS - MACAÉ - RJ,9618757,3813885801,MAR LIMPO II,0.0,13,0,13,...,0,"Desembarque/Embarque de Passageiros,Off-shore",2022-01-01 17:45:00-03:00,2022-01-03 06:50:00-03:00,2022-01-04 03:00:00-03:00,2022-01-04 17:40:00-03:00,BRRIO,RIO DE JANEIRO,BRMEA001,TERMINAL PETROBRÁS - MACAÉ - RJ


In [8]:
df_port_call.isna().mean().sort_values(ascending=False).head(20)

depature_port_ts         0.937458
unberthing_ts            0.937366
berthing_ts              0.934879
arrival_port_ts          0.934879
vessel_id                0.666519
imo                      0.097485
destination_port_name    0.031055
destination_port         0.031042
source_port              0.000309
port_call_reasons        0.000302
source_port_name         0.000296
                         0.000012
port_call_id             0.000000
vessel_name              0.000000
port                     0.000000
port_name                0.000000
                         0.000000
                         0.000000
                         0.000000
                         0.000000
dtype: float64

In [10]:
df_port_call

,port_call_id,port,port_name,imo,vessel_id,vessel_name,,,,,...,port_call_reasons,arrival_port_ts,berthing_ts,unberthing_ts,depature_port_ts,source_port,source_port_name,destination_port,destination_port_name,source_file
0,12022,BRSSZ,SANTOS,9634854,<NA>,ORION GLOBE,0.0,21,0,21,...,Carga,2022-01-13 06:00:00-03:00,2022-01-20 17:20:00-03:00,2022-01-23 21:52:00-03:00,2022-01-23 21:52:00-03:00,SGSIN,SINGAPURA (SINGAPORE),SGSIN,SINGAPURA (SINGAPORE),estadia_embarcacao.csv
1,22022,BRSSZ,SANTOS,9699359,<NA>,COPENHAGEN EAGLE,0.0,19,0,19,...,Descarga,2022-01-15 09:05:00-03:00,2022-02-25 05:05:00-03:00,2022-02-28 13:45:00-03:00,2022-02-28 13:45:00-03:00,SGSIN,SINGAPURA (SINGAPORE),BRPNG,PARANAGUA,estadia_embarcacao.csv
2,32022,BRSSZ,SANTOS,9686273,<NA>,CEPHEUS OCEAN,0.0,19,0,19,...,Descarga,2022-01-19 10:00:00-03:00,2022-02-28 20:25:00-03:00,NaT,NaT,RU006,UST - LUGA,<NA>,<NA>,estadia_embarcacao.csv
3,62022,BRRJ022,PORTO DO AÇU,9644158,<NA>,ZARAPITO,0.0,15,0,15,...,Off-shore,2022-01-01 08:00:00-03:00,2022-01-01 08:00:00-03:00,2022-01-03 10:27:00-03:00,2022-01-03 10:27:00-03:00,BRRIO,RIO DE JANEIRO,BRRIO,RIO DE JANEIRO,estadia_embarcacao.csv
4,72022,BRMEA001,TERMINAL PETROBRÁS - MACAÉ - RJ,9618757,3813885801,MAR LIMPO II,0.0,13,0,13,...,"Desembarque/Embarque de Passageiros,Off-shore",2022-01-01 17:45:00-03:00,2022-01-03 06:50:00-03:00,2022-01-04 03:00:00-03:00,2022-01-04 17:40:00-03:00,BRRIO,RIO DE JANEIRO,BRMEA001,TERMINAL PETROBRÁS - MACAÉ - RJ,estadia_embarcacao.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162031,610152025,BRNTR,NITERÓI,9763772,4430488649,STIM STAR BRASIL,0.0,17,0,17,...,Off-shore,NaT,NaT,NaT,NaT,BRMEA001,TERMINAL PETROBRÁS - MACAÉ - RJ,BRRJ022,PORTO DO AÇU,estadia_embarcacao.csv
162032,610182025,BRRIO,RIO DE JANEIRO,9759496,4430489777,STARNAV VOLANS,0.0,15,0,15,...,Off-shore,NaT,NaT,NaT,NaT,BRRIO,RIO DE JANEIRO,BRRIO,RIO DE JANEIRO,estadia_embarcacao.csv
162033,610312025,BRRIO,RIO DE JANEIRO,9357169,3810518310,ASSO VENTISETTE,0.0,14,0,14,...,Off-shore,NaT,NaT,NaT,NaT,BRRIO,RIO DE JANEIRO,BRRIO,RIO DE JANEIRO,estadia_embarcacao.csv
162034,610382025,BRRIO,RIO DE JANEIRO,9497127,4430138389,CAMPOS CONTENDER,0.0,16,0,16,...,Off-shore,NaT,NaT,NaT,NaT,BRNTR,NITERÓI,BRRIO,RIO DE JANEIRO,estadia_embarcacao.csv


In [6]:
date_cols = [
    "arrival_port_ts",
    "berthing_ts",
    "unberthing_ts",
    "departure_port_ts",
]

df_port_call[date_cols].head(10)

,arrival_port_ts,berthing_ts,unberthing_ts,departure_port_ts
0,2022-01-13 06:00:00-03:00,2022-01-20 17:20:00-03:00,2022-01-23 21:52:00-03:00,2022-01-23 21:52:00-03:00
1,2022-01-15 09:05:00-03:00,2022-02-25 05:05:00-03:00,2022-02-28 13:45:00-03:00,2022-02-28 13:45:00-03:00
2,2022-01-19 10:00:00-03:00,2022-02-28 20:25:00-03:00,NaT,NaT
3,2022-01-01 08:00:00-03:00,2022-01-01 08:00:00-03:00,2022-01-03 10:27:00-03:00,2022-01-03 10:27:00-03:00
4,2022-01-01 17:45:00-03:00,2022-01-03 06:50:00-03:00,2022-01-04 03:00:00-03:00,2022-01-04 17:40:00-03:00
5,2022-01-29 07:20:00-03:00,2022-01-29 07:20:00-03:00,2022-01-29 13:00:00-03:00,2022-01-29 13:00:00-03:00
6,2022-01-06 08:45:00-03:00,2022-01-06 08:45:00-03:00,2022-01-07 18:20:00-03:00,2022-01-07 18:20:00-03:00
7,2022-01-04 08:40:00-03:00,2022-01-04 08:40:00-03:00,2022-01-04 16:15:00-03:00,2022-01-04 16:15:00-03:00
8,2022-01-07 08:10:00-03:00,2022-01-07 19:00:00-03:00,2022-01-07 21:50:00-03:00,2022-01-07 21:50:00-03:00
9,2022-01-02 10:00:00-03:00,2022-01-02 10:00:00-03:00,2022-01-02 22:40:00-03:00,2022-01-02 22:40:00-03:00


In [8]:
df_port_call.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162036 entries, 0 to 162035
Data columns (total 23 columns):
 #   Column                 Non-Null Count   Dtype                    
---  ------                 --------------   -----                    
 0   port_call_id           162036 non-null  int64                    
 1   port                   162036 non-null  object                   
 2   port_name              162036 non-null  object                   
 3   imo                    146240 non-null  object                   
 4   vessel_id              54036 non-null   object                   
 5   vessel_name            162036 non-null  object                   
 6   boarding_crew          162034 non-null  float64                  
 7   transit_crew           162036 non-null  int64                    
 8   unboarding_crew        162036 non-null  int64                    
 9   total_crew             162036 non-null  int64                    
 10  boarding_passengers    162036 no